# DIY Copilot
## Lesson: Hello GenAI World

Every coding assistant – GitHub Copilot, Cursor, Claude Code – is, at its core, a program that sends **messages** to a large language model (LLM) and does something with the answer. In this lesson you will make your very first call to an LLM and build the smallest possible "coding assistant".

We proceed in 5 steps:

- Step 1: Set up the environment
- Step 2: Show available models
- Step 3: Configure the model
- Step 4: Set up the prompt
- Step 5: Set up the generation configuration

### Step 1: Set up the environment

In [1]:
# >>> COLAB SETUP <<<
# Placeholder: clone the workshop repository into the Colab workspace and make `lib` importable.
# When running locally, the following line is sufficient.
import sys
sys.path.append('../../../')

In [2]:
from litellm import completion, get_valid_models

from lib.environment import load_environment

# Load the API key (Colab secret or .env file) and the default models
default_models = load_environment()

### Step 2: Show available models
We use the **Gemini API** via [LiteLLM](https://docs.litellm.ai/). LiteLLM offers one unified (OpenAI-compatible) interface for more than 100 providers, so switching from Gemini to OpenAI, Anthropic or a local model is mostly a matter of changing the model name.

Let's ask the provider which models are available for our API key.

In [3]:
PROVIDER = "gemini"

valid_models = get_valid_models(check_provider_endpoint=True, custom_llm_provider=PROVIDER)

for model in valid_models:
    print(model)

gemini/gemini-2.5-flash
gemini/gemini-2.5-pro
gemini/gemini-2.5-flash-preview-tts
gemini/gemini-2.5-pro-preview-tts
gemini/gemma-4-26b-a4b-it
gemini/gemma-4-31b-it
gemini/gemini-flash-latest
gemini/gemini-flash-lite-latest
gemini/gemini-pro-latest
gemini/gemini-2.5-flash-lite
gemini/gemini-2.5-flash-image
gemini/gemini-3-flash-preview
gemini/gemini-3.1-pro-preview
gemini/gemini-3.1-pro-preview-customtools
gemini/gemini-3.1-flash-lite-preview
gemini/gemini-3.1-flash-lite
gemini/gemini-3-pro-image-preview
gemini/gemini-3-pro-image
gemini/nano-banana-pro-preview
gemini/gemini-3.1-flash-image-preview
gemini/gemini-3.1-flash-image
gemini/gemini-3.1-flash-lite-image
gemini/gemini-nano-banana-2.1
gemini/gemini-3.5-flash
gemini/gemini-3.5-flash-lite
gemini/gemini-omni-flash-preview
gemini/gemini-omni-1.1-flash
gemini/gemini-3.5-transcribe
gemini/gemini-3.6-flash
gemini/gemini-3.7-flash
gemini/gemini-3.8-flash
gemini/gemini-4-argon
gemini/lyria-3-clip-preview
gemini/lyria-3-pro-preview
gemini/l

### Step 3: Configure the model
LiteLLM expects model ids in the format `<provider>/<model>`. Note that the list above contains models for different tasks (text generation, embeddings, images, ...). There is no endpoint telling you which model is meant for which task – you have to check the provider documentation.

In [4]:
MODEL = "gemini-3.1-flash-lite"
MODEL_ID = f"{PROVIDER}/{MODEL}"

### Step 4: Set up the prompt
A request consists of a list of **messages**. Each message has a **role**:
* `system`: instructions that define the behaviour of the assistant. In a coding assistant, this is where the tool vendor puts its (often very long) instructions.
* `user`: the request of the user.
* `assistant`: previous answers of the model (we will need this later for conversations).

In [5]:
SYSTEM_PROMPT = "You are a senior Python developer. You answer concisely and always include a short code example."
USER_PROMPT = "How do I check in Python whether two time ranges overlap?"

### Step 5: Set up the generation configuration

In [6]:
# Set some generation parameters. We will learn more about them in the next lesson.
TEMPERATURE = 1.0
MAX_TOKENS = 1000

In [7]:
# Print out a summary of the input values / parameters
print(f'SYSTEM PROMPT used:\n {SYSTEM_PROMPT}')
print(f'USER PROMPT used:\n {USER_PROMPT}')
print(f'MODEL used:\n {MODEL_ID} (temperature = {TEMPERATURE}, max_tokens = {MAX_TOKENS})')

SYSTEM PROMPT used:
 You are a senior Python developer. You answer concisely and always include a short code example.
USER PROMPT used:
 How do I check in Python whether two time ranges overlap?
MODEL used:
 gemini/gemini-3.5-flash-lite (temperature = 1.0, max_tokens = 1000)


### Exercise 01: Call the model
Generate a response by calling the *completion* method from `litellm` (see imports). Use the *model*, *messages* and *generation configuration parameters* as arguments.

In [8]:
# TODO: Set up the messages: the system prompt for the role "system" and the user prompt for the role "user"
messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": USER_PROMPT},
]

# TODO: Generate a response by calling the "completion" method from litellm.
# Use the model, messages and generation configuration parameters as arguments.
response = completion(
    model=MODEL_ID,
    messages=messages,
    temperature=TEMPERATURE,
    max_tokens=MAX_TOKENS,
)

In [9]:
# Print the complete response of the model
print('RESPONSE of the model:')
print(response)

RESPONSE of the model:
ModelResponse(id='Uz3sarHRHu_jnsEPiqagiQc', created=1793867089, model='gemini-3.5-flash-lite', object='chat.completion', system_fingerprint=None, choices=[Choices(finish_reason='stop', index=0, message=Message(content='Two time ranges `[start1, end1]` and `[start2, end2]` overlap if and only if the maximum of their start times is less than or equal to the minimum of their end times.\n\nHere is a concise example:\n\n```python\ndef ranges_overlap(r1, r2):\n    # r1 and r2 are tuples of (start, end)\n    return max(r1[0], r2[0]) <= min(r1[1], r2[1])\n\n# Example usage:\nrange1 = (10, 20)\nrange2 = (15, 25)\nrange3 = (21, 30)\n\nprint(ranges_overlap(range1, range2))  # True  (15 to 20 overlaps)\nprint(ranges_overlap(range1, range3))  # False (No overlap)\n```', role='assistant', tool_calls=None, function_call=None, images=[], thinking_blocks=[], provider_specific_fields={'thought_signatures': ['EmAKXgFpFH0T4sh6ypQfAC4RXIvwhVYzDbGnUcegDhcBfHYlxTwQKVL+iCwqnHwdDH55ZSfbr

### Exercise 02: Inspect the response
Print only the message of the model. What is the `role` of the message?

Then have a look at the `usage` of the response. Every coding assistant counts tokens: they determine the **costs**, the **latency** and how much code fits into the **context window** of the model.

In [10]:
# TODO: Print the role of the message
print('ROLE of the message:')
print(response.choices[0].message.role)
print()

# TODO: Print the content of the message
print('CONTENT of the message:')
print(response.choices[0].message.content)

ROLE of the message:
assistant

CONTENT of the message:
Two time ranges `[start1, end1]` and `[start2, end2]` overlap if and only if the maximum of their start times is less than or equal to the minimum of their end times.

Here is a concise example:

```python
def ranges_overlap(r1, r2):
    # r1 and r2 are tuples of (start, end)
    return max(r1[0], r2[0]) <= min(r1[1], r2[1])

# Example usage:
range1 = (10, 20)
range2 = (15, 25)
range3 = (21, 30)

print(ranges_overlap(range1, range2))  # True  (15 to 20 overlaps)
print(ranges_overlap(range1, range3))  # False (No overlap)
```


In [11]:
# TODO: Print the number of input (prompt) tokens and output (completion) tokens
print(f"Input tokens:  {response.usage.prompt_tokens}")
print(f"Output tokens: {response.usage.completion_tokens}")

Input tokens:  31
Output tokens: 199


### Exercise 03: Shape your assistant with the system prompt
The same model behaves completely differently depending on the system prompt. Coding assistants use this to create different "modes" (e.g. *ask*, *edit*, *review*).

Write two system prompts and compare the answers to the same user prompt:
1. A **mentor** who explains the solution step by step to a junior developer.
2. A **code generator** that answers with Python code only – no explanations, no Markdown.

In [12]:
# TODO: Define the two system prompts
MENTOR_PROMPT = (
    "You are a patient mentor for junior Python developers. "
    "Explain the solution step by step in simple words before you show the code."
)
CODE_ONLY_PROMPT = (
    "You are a code generator. Answer with valid Python code only. "
    "Do not add explanations and do not wrap the code in Markdown code fences."
)

In [13]:
# TODO: Call the model with both system prompts and the same user prompt and print the answers
for name, system_prompt in [("MENTOR", MENTOR_PROMPT), ("CODE ONLY", CODE_ONLY_PROMPT)]:
    response = completion(
        model=MODEL_ID,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": USER_PROMPT},
        ],
        temperature=TEMPERATURE,
        max_tokens=MAX_TOKENS,
    )
    print(f"========== {name} ==========")
    print(response.choices[0].message.content)
    print()

========== MENTOR ==========
Hello there! I'm happy to help you figure this out. 

Working with time ranges is a very common task in programming (like booking a meeting room or scheduling events). 

To figure out if two time ranges overlap, it helps to think about the **opposite**: when do they *not* overlap? 
Two ranges will **not** overlap if:
1. Range A completely finishes *before* Range B even starts.
2. Range A completely starts *after* Range B has already finished.

If neither of those "non-overlapping" situations is true, then **they must overlap!**

Let's break this down into steps using Python.

### Step 1: Represent our time ranges
For simplicity, let's represent times as simple numbers (like hours in a 24-hour day, e.g., 9 AM is `9`, 5 PM is `17`). 
Let's say:
* Range 1 starts at `start1` and ends at `end1`
* Range 2 starts at `start2` and ends at `end2`

### Step 2: Write down the "non-overlap" rule
Range 1 is completely before Range 2 if `end1 <= start2`.
Range 1 is comple

### Default models
To avoid configuring the model in every notebook, the following notebooks use the default models returned by `load_environment()`. You can override them by adding the secrets `DEFAULT_GENERATION_MODEL` and `DEFAULT_EMBEDDING_MODEL` (Colab) or the corresponding entries in your `.env` file (local).

In [14]:
print(f"Default model for generation: {default_models['generation_model']}")
print(f"Default model for embeddings: {default_models['embedding_model']}")

Default model for generation: gemini-2.5-flash
Default model for embeddings: qwen3-embedding:0.6b
